# PEPA 2026: Clase 2 - Desarrollar la intuición sobre la variabilidad y el azar.

Este Notebook está diseñado para ser corrido en **Google Colab**. Explora los conceptos de la Clase 2 (Estadística Descriptiva y Probabilidad) utilizando herramientas interactivas.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Layout, VBox, Output
from IPython.display import display

# Configuración estética
%matplotlib inline
plt.style.use('seaborn-v0_8-muted')

## 1. El Peligro de los Outliers en SRE (Site Reliability Engineering)

Imagina que mides la latencia (en milisegundos) de las peticiones a tu servidor. La mayoría se procesan en $\sim 15$ms, pero ¿qué pasa si una sola petición se cuelga (outlier)?

Juega con el deslizador del **Outlier** y observa cómo reaccionan la **Media** y la **Mediana**.

In [ ]:
def graficar_latencias(outlier_val):
    # Datos normales (latencia estable)
    datos_base = np.array([12, 15, 11, 14, 13, 16, 12, 13, 15, 14])
    # Agregamos el outlier interactivo
    datos = np.append(datos_base, outlier_val)

    media = np.mean(datos)
    mediana = np.median(datos)

    fig, ax = plt.subplots(figsize=(10, 2))

    # Scatter plot de los datos
    ax.scatter(datos, np.zeros_like(datos), color='gray', s=100, alpha=0.6, label='Latencias')
    ax.scatter(outlier_val, 0, color='red', s=150, zorder=5, label='Outlier (Timeout)')

    # Líneas de tendencia central
    ax.axvline(media, color='blue', linestyle='--', label=f'Media: {media:.1f}ms')
    ax.axvline(mediana, color='green', linestyle='-', label=f'Mediana: {mediana:.1f}ms')

    ax.set_yticks([])
    ax.set_xlim(0, max(100, outlier_val + 20))
    ax.set_title("Impacto de un Outlier (Timeout) en Métricas SRE")
    ax.set_xlabel("Latencia (ms)")
    ax.legend()
    plt.show()

interact(graficar_latencias, outlier_val=FloatSlider(value=15, min=10, max=500, step=10, description='Outlier (ms):'));

interactive(children=(FloatSlider(value=15.0, description='Outlier (ms):', max=500.0, min=10.0, step=10.0), Ou…

### 1.1 El Secreto Matemático: Funciones de pérdida $L_1$ vs $L_2$

¿Por qué la mediana no se mueve y la media sí? La matemática nos da una respuesta elegante. Estos estadísticos son en realidad el valor $c$ que **minimiza diferentes funciones de error**:

* **La Media ($\bar{x}$)** es el valor $c$ que minimiza el **Error Cuadrático ($L_2$)**: $\sum (x_i - c)^2$. Al elevar al cuadrado, las distancias grandes reciben una penalización gigantesca. La media es *arrastrada* hacia el outlier para reducir ese enorme error.
* **La Mediana** es el valor $c$ que minimiza el **Error Absoluto ($L_1$)**: $\sum |x_i - c|$. Como la distancia es lineal, un outlier a $1000$ unidades de distancia empuja a la mediana exactamente con la misma fuerza que si estuviera a $10$ unidades.

Veámoslo gráficamente. Observa cómo las curvas de error $L_1$ (azul) y $L_2$ (rojo) reaccionan ante un outlier. ¡Fíjate dónde caen sus mínimos!

In [ ]:
def plot_funciones_perdida(outlier_val):
    # Datos base sencillos + 1 outlier interactivo
    datos = np.array([10, 12, 14, 16, outlier_val])

    # Rango de posibles candidatos a "centro" (c)
    c_vals = np.linspace(8, max(30, outlier_val), 500)

    # Calculamos los errores L1 y L2 para cada candidato c
    L1 = [np.sum(np.abs(datos - c)) for c in c_vals]
    L2 = [np.sum((datos - c)**2) for c in c_vals]

    media = np.mean(datos)
    mediana = np.median(datos)

    fig, ax1 = plt.subplots(figsize=(10, 5))

    # Eje para L1 (Mediana)
    color1 = 'tab:blue'
    ax1.set_xlabel('Valor candidato a centro (c)')
    ax1.set_ylabel('Suma de Errores Absolutos ($L_1$)', color=color1)
    ax1.plot(c_vals, L1, color=color1, label='$L_1$ (Minimizado por Mediana)')
    ax1.axvline(mediana, color='blue', linestyle=':', linewidth=3, label=f'Mínimo L1: Mediana ({mediana})')
    ax1.tick_params(axis='y', labelcolor=color1)

    # Eje secundario para L2 (Media)
    ax2 = ax1.twinx()
    color2 = 'tab:red'
    ax2.set_ylabel('Suma de Errores Cuadráticos ($L_2$)', color=color2)
    ax2.plot(c_vals, L2, color=color2, label='$L_2$ (Minimizado por Media)')
    ax2.axvline(media, color='red', linestyle='--', linewidth=3, label=f'Mínimo L2: Media ({media:.1f})')
    ax2.tick_params(axis='y', labelcolor=color2)

    plt.title(f"Mínimos de $L_1$ y $L_2$ con Outlier ubicado en {outlier_val}")
    fig.legend(loc="upper center", bbox_to_anchor=(0.5, 0.85))
    plt.show()

interact(plot_funciones_perdida,
         outlier_val=IntSlider(min=16, max=100, step=2, value=16,
                               description='Outlier', layout=Layout(width='80%')));

interactive(children=(IntSlider(value=16, description='Outlier', layout=Layout(width='80%'), min=16, step=2), …

## 2. La Ley de los Grandes Números (Simulación Frecuentista)

Si la probabilidad teórica de sacar un '6' en un dado equilibrado es $1/6 \approx 0.1667$, ¿qué pasa en la realidad si lanzamos el dado pocas veces vs miles de veces?

Mueve el deslizador para aumentar el número de lanzamientos y observa cómo la frecuencia empírica oscila al principio, pero luego converge hacia la probabilidad estimada.

In [ ]:
def simular_probabilidad(n_lanzamientos):
    tiros = np.random.randint(1, 7, n_lanzamientos)

    # Calculamos la probabilidad acumulada de que salga un '6'
    es_seis = (tiros == 6).astype(float)
    prob_acumulada = np.cumsum(es_seis) / np.arange(1, n_lanzamientos + 1)

    plt.figure(figsize=(12, 5))
    plt.plot(prob_acumulada, color='orange', label='Probabilidad estimada ($n_6/n$)')
    plt.axhline(1/6, color='red', linestyle='--', label='Probabilidad teórica (1/6)')

    plt.title(f"Convergencia tras {n_lanzamientos} lanzamientos")
    plt.xlabel("Número de experimentos (n)")
    plt.ylabel("Probabilidad")
    plt.ylim(0, 0.4)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

interact(simular_probabilidad, n_lanzamientos=IntSlider(value=10, min=10, max=10000, step=100, description='N tiros:'));

interactive(children=(IntSlider(value=10, description='N tiros:', max=10000, min=10, step=100), Output()), _do…

---

## 🧪 Ejercicios de Exploración

### Ejercicio 1: La oscilación de $\hat{p}$ como función de $n$

En la simulación viste que la frecuencia relativa converge a $1/6$. Pero, ¿cuánto oscila entre una corrida y otra?

**Consigna:** Para cada $n \in \{10, 50, 100, 500, 1000, 5000, 10000\}$, repetí el experimento de lanzar el dado $n$ veces **100 veces**, y calculá el desvío estándar $s$ de las 100 frecuencias relativas obtenidas. Graficá $s$ en función de $n$.

1. ¿Qué forma tiene la curva? ¿Crece, decrece, es lineal?
2. ¿Qué le pasa a $s$ si cuadruplicás $n$?
3. Superponé la curva $\frac{1}{2\sqrt{n}}$ sobre tu gráfico. ¿Coincide con lo simulado?

---

### Ejercicio 2: Histogramas de $\hat{p}$ para distintos $n$

**Consigna:** Simulá 10.000 experimentos de lanzar una moneda $n$ veces para cada $n \in \{10, 50, 200, 1000\}$. Graficá los cuatro histogramas de **frecuencia relativa** $\hat{p} = k/n$ en una misma figura con `plt.subplots`, fijando el eje $x$ entre 0 y 1 en todos los paneles.

1. ¿Qué le pasa al ancho del histograma cuando $n$ crece?
2. ¿Qué le pasa a su forma?
3. Calculá el desvío estándar $s$ de los 10.000 valores de $\hat{p}$ para cada $n$. ¿Cómo se relaciona con $\frac{1}{2\sqrt{n}}$?

---

### Ejercicio 3: Transformaciones lineales — verificación interactiva

La teoría dice que para $y_i = a + b\, x_i$:
$$\bar{y} = a + b\bar{x} \qquad \text{y} \qquad s^2_y = b^2\, s^2_x$$

**Consigna:** Usando las latencias del servidor de la Sección 1 (con el outlier fijo en 15 ms), construí un widget con sliders para $a$ y $b$ que muestre en tiempo real:

- Los histogramas de $x_i$ e $y_i$ lado a lado.
- Los valores de $\bar{x}$, $s^2_x$, $\bar{y}$, $s^2_y$ junto a las predicciones teóricas $a + b\bar{x}$ y $b^2 s^2_x$.

Respondé:

1. ¿Coinciden siempre los valores calculados con los predichos?
2. ¿Qué efecto tiene $a$ sobre el histograma? ¿Y sobre $s^2$?
3. ¿Qué pasa si $b < 0$? ¿Cambia $s^2_y$?
4. Si las latencias están en milisegundos y querés pasarlas a segundos, ¿qué $a$ y $b$ usarías? ¿Cómo cambian $\bar{x}$ y $s$?